<a href="https://colab.research.google.com/github/buiky5478-collab/buiky/blob/main/baitap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_timestamp, sum

# Khởi tạo Spark
spark = SparkSession.builder \
    .appName("DataLakePipeline") \
    .getOrCreate()

# --- CHUẨN BỊ ĐƯỜNG DẪN ---
datalake_path = '/content/drive/MyDrive/DataLake_Lab'
bronze_path = f'{datalake_path}/Raw_Zone/sales_data.csv'
silver_path = f'{datalake_path}/Silver_Zone/sales_cleaned'
gold_path = f'{datalake_path}/Gold_Zone/country_revenue_summary'

# ==========================================
# CÂU 1: TẠO TẦNG SILVER
# ==========================================
print("⏳ 1. Đang xử lý tầng Silver...")

df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)

df_silver = df_bronze.filter(col("CustomerID").isNotNull()) \
                     .filter(~col("InvoiceNo").startswith("C")) \
                     .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")) \
                     .withColumn("Year", year(col("InvoiceDate"))) \
                     .withColumn("TotalAmount", col("Quantity") * col("UnitPrice"))

df_silver.write.mode("overwrite").parquet(silver_path)

# ==========================================
# CÂU 2: TẠO TẦNG GOLD
# ==========================================
print("⏳ 2. Đang xử lý tầng Gold...")

df_clean = spark.read.parquet(silver_path)

df_gold = df_clean.groupBy("Country", "Year") \
                  .agg(sum("TotalAmount").alias("TotalRevenue"))

df_gold.write \
       .partitionBy("Year") \
       .mode("overwrite") \
       .parquet(gold_path)

# ==========================================
# CÂU 3: BÁO CÁO 2011
# ==========================================
print("✅ 3. Hoàn tất Pipeline! Đang lấy báo cáo năm 2011...")

df_report_2011 = spark.read.parquet(f"{gold_path}/Year=2011")

df_report_2011.orderBy(col("TotalRevenue").desc()).show(5)

Mounted at /content/drive
⏳ 1. Đang xử lý tầng Silver...
⏳ 2. Đang xử lý tầng Gold...
✅ 3. Hoàn tất Pipeline! Đang lấy báo cáo năm 2011...
+--------------+------------------+
|       Country|      TotalRevenue|
+--------------+------------------+
|United Kingdom| 6809729.704000123|
|   Netherlands| 276661.8599999993|
|          EIRE| 256732.0199999992|
|       Germany|213626.00000000023|
|        France|199407.74000000017|
+--------------+------------------+
only showing top 5 rows
